# Pretraining objectives: MLM, next-token, denoising

Pretraining turns raw text into labels by hiding, shifting, or corrupting tokens.

The architecture matters, but the objective decides which positions receive loss. Masked, shifted, and denoising labels produce different supervision sets and should be compared carefully. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(94)


def nll(probabilities):
    return -sum(math.log(max(float(p), 1e-12)) for p in probabilities)


def pretraining_ladder():
    base = ["ad", "creative", "with", "fresh", "video", "gets", "more", "clicks", "from", "mobile", "members", "during", "launch", "week", "when", "copy", "is", "clear", "and", "short"]
    return [
        {"name": "D1 one 20-token sentence", "tokens": base, "mask_rate": 0.15, "probs": [0.8, 0.6, 0.5], "objective": "mlm"},
        {"name": "D2 clean synthetic corpus", "tokens": base[:12], "mask_rate": 0.15, "probs": [0.75, 0.65, 0.55], "objective": "mlm"},
        {"name": "D3 high-corruption noisy corpus", "tokens": base[:16], "mask_rate": 0.35, "probs": [0.52, 0.48, 0.44, 0.40, 0.38], "objective": "denoise"},
        {"name": "D4 tiny real text snippets", "tokens": ["campaign", "manager", "shows", "low", "quality", "creative", "needs", "clear", "hook"], "mask_rate": 0.22, "probs": [0.62, 0.54], "objective": "next"},
        {"name": "D5 long objective comparison", "tokens": base + ["audience", "fit", "matters", "for", "return"], "mask_rate": 0.40, "probs": [0.5, 0.48, 0.42, 0.39, 0.36, 0.33], "objective": "mixed"},
    ]


def mask_positions(length, mask_rate):
    count = max(1, int(round(length * mask_rate)))
    return list(range(1, min(length, count + 1)))

## The concept, built once: objective-specific loss

Pretraining chooses which token positions receive loss:
$$\mathcal{L}= -\sum_{t\in M}\log p(x_t\mid x_{\setminus M})\quad\text{or}\quad -\sum_t\log p(x_t\mid x_{\lt t}).$$
The same text can therefore create MLM labels, next-token labels, or denoising reconstruction targets.

In [ ]:
def pretrain_loss(tokens, objective, probabilities, mask_rate=0.15, span_length=5):
    positions = mask_positions(len(tokens), mask_rate)
    if objective == "mlm":
        supervised = positions
    elif objective == "next":
        supervised = list(range(1, len(tokens)))
    elif objective == "denoise":
        supervised = positions
    else:
        supervised = positions + list(range(1, min(len(tokens), len(positions) + 2)))
    loss = nll(probabilities)
    compressed_length = len(tokens) - span_length + 1
    return {
        "loss": loss,
        "supervised": supervised,
        "compressed_length": compressed_length,
    }

The lesson checks MLM loss $-\log0.8-\log0.6-\log0.5=1.427$, next-token loss on $0.7,0.5,0.2$ equals 2.659, 20 tokens at 15% masking create 3 labels, replacing 5 tokens by one sentinel shortens 20 to 16, and $0.75(1.4)+0.25(2.0)=1.55$.

In [ ]:
tokens = pretraining_ladder()[0]["tokens"]
mlm_loss = pretrain_loss(tokens, "mlm", [0.8, 0.6, 0.5])["loss"]
next_loss = pretrain_loss(tokens, "next", [0.7, 0.5, 0.2])["loss"]
label_count = int(0.15 * 20)
compressed = pretrain_loss(tokens, "denoise", [0.8], span_length=5)["compressed_length"]
mixed = 0.75 * 1.4 + 0.25 * 2.0

assert round(mlm_loss, 3) == 1.427
assert round(next_loss, 3) == 2.659
assert label_count == 3
assert compressed == 16
assert round(mixed, 2) == 1.55

print("MLM loss", round(mlm_loss, 3))
print("next-token loss", round(next_loss, 3))
print("labels", label_count)
print("compressed length", compressed)
print("mixed loss", round(mixed, 2))

## The dataset ladder

D1-D5 varies corruption rate, objective type, and supervised-token count so losses can be normalized rather than compared naively.

In [ ]:
ladder = pretraining_ladder()
for rung in ladder:
    positions = mask_positions(len(rung["tokens"]), rung["mask_rate"])
    print(rung["name"], "tokens", len(rung["tokens"]), "objective", rung["objective"], "labels", len(positions), "sample", rung["tokens"][:6])

In [ ]:
rows = []
for rung in ladder:
    result = pretrain_loss(rung["tokens"], rung["objective"], rung["probs"], mask_rate=rung["mask_rate"])
    token_nll = result["loss"] / max(1, len(rung["probs"]))
    rows.append({
        "name": rung["name"],
        "corruption": rung["mask_rate"],
        "token_nll": token_nll,
        "supervised": len(result["supervised"]),
        "objective": rung["objective"],
    })
for row in rows:
    print(f"{row['name']}: objective={row['objective']} supervised={row['supervised']} token_NLL={row['token_nll']:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, row in enumerate(rows):
    axes[0, idx].bar(["supervised", "hidden"], [row["supervised"], max(1, 20 - row["supervised"])], color=["crimson", "lightgray"])
    axes[0, idx].set_title(f"D{idx + 1} {row['objective']}")
    axes[1, idx].bar(["NLL"], [row["token_nll"]], color="crimson")
fig.suptitle("Target panels and normalized NLL per rung")
plt.figure(figsize=(6, 3))
plt.plot([row["corruption"] for row in rows], [row["token_nll"] for row in rows], marker="o")
plt.xlabel("corruption rate")
plt.ylabel("supervised-token NLL")
plt.grid(True)

## Pitfall on D5: comparing objectives without token-set normalization

MLM and next-token objectives supervise different positions. The fix is to report loss per supervised token and keep the objective name attached to the number.

In [ ]:
hard = ladder[-1]
mlm_total = nll([0.50, 0.48, 0.42])
next_total = nll([0.70, 0.68, 0.64, 0.60, 0.58, 0.55, 0.52, 0.50])
misleading = "MLM looks better" if mlm_total < next_total else "next-token looks better"
mlm_per_token = mlm_total / 3
next_per_token = next_total / 8
fixed = "MLM lower per token" if mlm_per_token < next_per_token else "next-token lower per token"
print("raw totals", round(mlm_total, 3), round(next_total, 3), misleading)
print("per-token normalized", round(mlm_per_token, 3), round(next_per_token, 3), fixed)

## Evaluate it + practice

            - Metric: supervised-token negative log-likelihood; compare to the no-skill baseline, predict the unigram distribution at every supervised position.
            - Cheap sanity check: label counts match the corruption or shift rule.
            - Ablation: count unmasked prompt tokens in the loss and watch the objective change.
            - Failure signals: raw totals rank objectives but per-token loss reverses or changes interpretation.
            - Practice: Change D3 corruption from 35% to 50% and count labels.
- Practice: Compute the denoising compressed length for a 7-token span.
- Practice: Normalize two objective losses by supervised-token count.